In [ ]:
# to run on collab, simply execute the following commands in a cell:
# !git clone https://github.com/Torpedoooo/deeplearning-assignment-1.git
# !pip install torch torchvision pandas seaborn scikit-learn
# !pip install nbconvert
# %cd deeplearning-assignment-1/
# !jupyter nbconvert --execute task2.ipynb \
#                     --to notebook --output executed.ipynb


# Task 2: Convolutional Neural Network (CNN) Classification

### Overview
In this second stage, we transition from a Multilayer Perceptron (MLP) to a Convolutional Neural Network (CNN). While MLPs require flattening images into 1D arrays—destroying all spatial relationships—CNNs slide 2D filters across the image. This allows the network to learn spatial hierarchies, such as recognizing edges, textures, and ultimately physical Pokémon features. 

We will also rigorously track our computational resources (Training Time and GPU Memory) to strictly adhere to the assignment's compute budget constraints.

In [ ]:
import os
import time
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### Dataset Setup
We utilize the same `CustomImageDataset` class from Task 1 to load our `.png` files and map the string labels (e.g., 'Water', 'Fire') to numeric indices.

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir,
                 transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        classes = sorted(self.img_labels.iloc[:,1].unique())
        self.class2idx = {c:i for i,c in enumerate(classes)}
        self.img_dir    = img_dir
        self.transform  = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_name = self.img_labels.iloc[idx, 0]
        if not img_name.lower().endswith('.png'):
            img_name = img_name + '.png'
        label = self.img_labels.iloc[idx, 1]
        label = self.class2idx[label]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)

        return image, label

### Data Splitting & Augmentation
We split our dataset using an 80/20 random sample. To artificially expand our training data and combat overfitting, we apply a `RandomHorizontalFlip`. Images are resized to 64x64 and normalized using standard ImageNet statistics.

In [ ]:
# Read full data
df = pd.read_csv('train_labels.csv')

# Random 80/20 split
train_df = df.sample(frac=0.8, random_state=42)
test_df  = df.drop(train_df.index)

train_df.to_csv('train_split.csv', index=False)
test_df.to_csv('test_split.csv',  index=False)

print("train:", len(train_df), "rows")
print("test :", len(test_df),  "rows")

# Transformations
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

### Handling Class Imbalance
Because our dataset is heavily skewed (e.g., 'Water' appears far more often than 'Ground' or 'Rock'), we deploy a `WeightedRandomSampler`. This ensures that minority classes are oversampled during training so the model receives balanced batches.

In [ ]:
train_csv = './train_split.csv'
test_csv  = './test_split.csv'
train_dir = './Train/'

train_ds = CustomImageDataset(train_csv, train_dir, transform=train_transform)
test_ds  = CustomImageDataset(test_csv, train_dir, transform=val_transform)

# Class-level weights for the sampler
labels = train_ds.img_labels['label'].map(train_ds.class2idx)
class_counts  = labels.value_counts().sort_index()
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()

sample_weights = labels.map(class_weights).values
sample_weights = torch.tensor(sample_weights, dtype=torch.double)

sampler = WeightedRandomSampler(
    weights = sample_weights,
    num_samples = len(sample_weights),
    replacement = True
)

# Initialize DataLoaders
train_loader = DataLoader(train_ds, batch_size=64, sampler=sampler)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

### CNN Architecture
Our custom CNN features three convolutional blocks:
* **Convolutions:** Channels progressively increase (32 -> 64 -> 128) to learn increasingly complex features.
* **Pooling:** `MaxPool2d` reduces the spatial dimensions, lowering the parameter count and computation cost.
* **Classifier:** The extracted features are flattened and passed through a fully connected layer. We apply 50% `Dropout` here to aggressively mitigate overfitting.

In [ ]:
class PokemonCNN(nn.Module):
    def __init__(self, num_classes=9):
        super(PokemonCNN, self).__init__()
        
        # Conv Block 1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # 64x64 -> 32x32
        
        # Conv Block 2
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # 32x32 -> 16x16
        
        # Conv Block 3
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2) # 16x16 -> 8x8
        
        # Fully Connected Layers
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128 * 8 * 8, 512)
        self.relu4 = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        
        x = self.flatten(x)
        x = self.dropout(self.relu4(self.fc1(x)))
        x = self.fc2(x)
        return x

cnn_model = PokemonCNN(num_classes=9).to(device)
criterion = nn.CrossEntropyLoss()
print(cnn_model)

### Training Loop & Resource Monitoring
To respect the strict 1-hour compute budget for the CNN task, we use Early Stopping based on the Validation **Macro F1-Score**. This halts training when the model stops learning balanced representations of all classes. 

Additionally, we use Python's `time` library and `torch.cuda` memory stats to log our exact compute footprint for reporting.

In [ ]:
cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=1e-3)
cnn_scheduler = torch.optim.lr_scheduler.StepLR(cnn_optimizer, step_size=15, gamma=0.5)

def batch_accuracy(logits, labels):
    preds = logits.argmax(dim=1)
    return (preds == labels).float().mean().item()

best_cnn_f1 = 0.0
trigger_times = 0
patience = 20
epochs = 100

# --- RESOURCE TRACKING START ---
start_time = time.time()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


for epoch in range(1, epochs+1):
    cnn_model.train()
    running_loss, running_acc = 0.0, 0.0
    
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        logits = cnn_model(xb)
        loss = criterion(logits, yb)

        cnn_optimizer.zero_grad()
        loss.backward()
        cnn_optimizer.step()

        running_loss += loss.item() * xb.size(0)
        running_acc  += batch_accuracy(logits, yb) * xb.size(0)
        
    cnn_scheduler.step()
    train_loss = running_loss / len(train_ds)
    train_acc  = running_acc  / len(train_ds)

    # Validation
    cnn_model.eval()
    val_loss, val_acc = 0.0, 0.0
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = cnn_model(xb)
            
            val_loss += criterion(logits, yb).item() * xb.size(0)
            val_acc  += batch_accuracy(logits, yb) * xb.size(0)
            
            preds = logits.argmax(dim=1)
            all_preds.append(preds.cpu())
            all_targets.append(yb.cpu())
            
    final_preds   = torch.cat(all_preds)
    final_targets = torch.cat(all_targets)
    current_macro_f1 = f1_score(final_targets, final_preds, average='macro')
    
    val_loss /= len(test_ds)
    val_acc  /= len(test_ds)
    
    if current_macro_f1 > best_cnn_f1:
        best_cnn_f1 = current_macro_f1
        trigger_times = 0
        torch.save(cnn_model.state_dict(), 'best_cnn_model.pth')
        print(f"Epoch {epoch:2d}: New best CNN f1_score ({best_cnn_f1:.3f}) Saving model")
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print(f"Epoch {epoch:2d}: Early stopping triggered.")
            break

# --- RESOURCE TRACKING END ---
end_time = time.time()
training_time_mins = (end_time - start_time) / 60

print("-" * 50)
print(f"CNN Training Completed in {training_time_mins:.2f} minutes.")
if torch.cuda.is_available():
    max_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    print(f"Max GPU Memory Allocated: {max_mem_mb:.2f} MB")
print("-" * 50)

### Evaluation & MLP Comparison
By generating a classification report and a confusion matrix on the test set, we can directly compare this CNN to our previous MLP. The spatial awareness of the convolutional filters should yield noticeably better predictive power across all 9 Pokémon types.

In [ ]:
# Load the best CNN model
cnn_model.load_state_dict(torch.load('best_cnn_model.pth'))
cnn_model.eval()

all_cnn_preds, all_cnn_targets = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = cnn_model(xb)
        preds = logits.argmax(dim=1)
        all_cnn_preds.append(preds.cpu())
        all_cnn_targets.append(yb.cpu())

all_cnn_preds   = torch.cat(all_cnn_preds)
all_cnn_targets = torch.cat(all_cnn_targets)

# Accuracy & Report
cnn_accuracy = (all_cnn_preds == all_cnn_targets).float().mean().item()
print(f"CNN Validation Accuracy: {cnn_accuracy:.4f}\n")

type_names = sorted(train_ds.class2idx, key=lambda k: train_ds.class2idx[k])
print(classification_report(all_cnn_targets, all_cnn_preds, target_names=type_names))

# Confusion Matrix Heatmap
cm_cnn = confusion_matrix(all_cnn_targets, all_cnn_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm_cnn, annot=True, fmt="d", xticklabels=type_names, yticklabels=type_names, cmap="Greens")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("CNN Confusion Matrix")
plt.show()

In [ ]:
# Map numeric indices back to string labels
idx2class = {v:k for k,v in train_ds.class2idx.items()}

# Get sorted test IDs so they match the Kaggle grading key
test_ids = sorted([f.split('.png')[0] for f in os.listdir('Test') if f.endswith('.png')])

class ImageOnlyDataset(Dataset):
    def __init__(self, ids, img_dir, transform=None):
        self.ids = ids
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        name = self.ids[idx]
        path = os.path.join(self.img_dir, name + '.png')
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img

# Load the test dataset
submission_ds = ImageOnlyDataset(test_ids, 'Test/', transform=val_transform)
submission_loader = DataLoader(submission_ds, batch_size=64, shuffle=False)

# Prediction loop
cnn_model.eval()
submission_preds = []

with torch.no_grad():
    for xb in submission_loader:
        xb = xb.to(device)
        logits = cnn_model(xb)
        submission_preds.extend(logits.argmax(1).cpu().tolist())

print("Test prediction distribution:", torch.bincount(torch.tensor(submission_preds)))
assert len(submission_preds) == len(test_ids)

# Map back to strings and write submission file
out_df = pd.DataFrame({
    'Id': test_ids,
    'label': [idx2class[i] for i in submission_preds]
})
out_df.to_csv('submission_cnn.csv', index=False)